# Stage 2: Data Preparation

This notebook performs missing-value handling, categorical encoding, scaling, and train-test splitting. ANN modeling is not included in Stage 2.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("../Dataset_ATS_v2.csv") if Path("../Dataset_ATS_v2.csv").exists() else Path("Dataset_ATS_v2.csv")
OUTPUT_DIR = Path(".")
RANDOM_STATE = 42

In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
for column in [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]:
    df[column] = df[column].astype(str).str.strip()

df["tenure"] = pd.to_numeric(df["tenure"], errors="coerce")
df["MonthlyCharges"] = pd.to_numeric(df["MonthlyCharges"], errors="coerce")
df["SeniorCitizen"] = pd.to_numeric(df["SeniorCitizen"], errors="coerce").astype("Int64")

print(df.shape)
display(df.head())
display(df.isna().sum())

In [ ]:
prepared_df = df.copy()
for column in ["SeniorCitizen", "tenure", "MonthlyCharges"]:
    prepared_df[column] = prepared_df[column].fillna(prepared_df[column].median())
for column in [col for col in prepared_df.columns if pd.api.types.is_string_dtype(prepared_df[col])]:
    prepared_df[column] = prepared_df[column].fillna(prepared_df[column].mode()[0])

y = prepared_df["Churn"].map({"No": 0, "Yes": 1}).astype(int)
numeric_features = ["SeniorCitizen", "tenure", "MonthlyCharges"]
categorical_features = [col for col in prepared_df.columns if col not in numeric_features + ["Churn"]]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), categorical_features),
])

x_encoded = preprocessor.fit_transform(prepared_df.drop(columns=["Churn"]))
feature_names = numeric_features + list(preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features))
encoded_df = pd.DataFrame(x_encoded, columns=feature_names)
encoded_df["Churn"] = y
display(encoded_df.head())

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    encoded_df.drop(columns=["Churn"]),
    encoded_df["Churn"],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=encoded_df["Churn"],
)

encoded_df.to_csv(OUTPUT_DIR / "preprocessed_dataset.csv", index=False)
X_train.to_csv(OUTPUT_DIR / "X_train.csv", index=False)
X_test.to_csv(OUTPUT_DIR / "X_test.csv", index=False)
Y_train.to_frame("Churn").to_csv(OUTPUT_DIR / "Y_train.csv", index=False)
Y_test.to_frame("Churn").to_csv(OUTPUT_DIR / "Y_test.csv", index=False)
joblib.dump(preprocessor, OUTPUT_DIR / "preprocessing_pipeline.joblib")

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Y_train:", Y_train.shape)
print("Y_test:", Y_test.shape)